由於訓練是在 Colab GPU 環境執行，請確認 Google Drive 中的專案目錄（`Pick Up Pan Test/`）包含 `datasets.zip`（內含 `datasets/so101_pick_stylus` 的 `data/`、`videos/`、`meta/`）。



> **架構說明（Fully Connected Network / MLP）**：本 Notebook 將原本的 SmolVLA 改為純全連接網路（`FCActionPolicy`，由多層 `nn.Linear` + `LayerNorm` + `GELU` + `Dropout` 組成），將雙相機影像降採樣展平後與 6 軸關節狀態 `observation.state` 串接，直接預測未來 50 步的 Action Chunk。

> **斷點續訓與自動存檔說明**：本 Notebook 保留完整斷點續訓機制（包含模型權重、設定檔、Optimizer、LR Scheduler、Loss 紀錄與隨機種子），預設每 **500 steps** 透過本地暫存與原子同步更新 Google Drive 的 `output_fc/checkpoints/latest`（與 SmolVLA 的 `output/checkpoints` 隔離不覆蓋）。

# 環境掛載與套件安裝
掛載 Google Drive 並安裝 `lerobot[dataset]` 與 `av`（不需下載 SmolVLM2 預訓練權重）。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q "lerobot[dataset]" av
try:
    import lerobot.utils.import_utils as _iu
    _iu._require_package_cache.clear()
except Exception:
    pass

In [ ]:
import gc
import glob
import json
import math
import os
import random
import shutil
import time

os.environ["MUJOCO_GL"] = "egl"
os.environ["PYOPENGL_PLATFORM"] = "egl"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from lerobot.datasets.lerobot_dataset import LeRobotDataset, LeRobotDatasetMetadata

if not os.path.ismount("/content/drive"):
    raise RuntimeError("Google Drive 尚未掛載，請先執行上方 drive.mount('/content/drive') Cell！")

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device, "| torch:", torch.__version__)

# 載入實體 SO-101 雙相機資料集、建立 Fully Connected Network (MLP) 並偵測斷點
1. **全連接輸入編碼**：將 `observation.images.top` 與 `observation.images.front` 降採樣為 `32x32` 後展平（可於參數設為 `FC_USE_IMAGES = False` 改為純關節狀態輸入），與 `observation.state` 串接後輸入純全連接網路。
2. **MEAN_STD 正規化**：直接讀取 `ds.meta.stats` 並註冊為模型內部 Buffer，訓練時對 `observation.state`、影像與 `action` 做正規化，推論時透過 `policy.predict_action(...)` 自動反正規化回 SO-101 關節角度。
3. **遮罩損失與斷點續訓**：訓練 Loss 使用 `action_is_pad` 遮罩過濾 episode 結尾補齊幀；若 `output_fc/checkpoints/` 已有完整 checkpoint，會自動接續訓練。

In [ ]:
if not os.path.ismount("/content/drive"):
    raise RuntimeError("Google Drive 尚未掛載，請先執行上方 drive.mount('/content/drive') Cell！")

PROJECT_DIR = "/content/drive/MyDrive/學校/交大/01_課程/04_Physical AI/Pick Up Pan Test"
DS_ZIP = os.path.join(PROJECT_DIR, "datasets.zip")
REPO_ID = "local/so101_pick_stylus"
DS_ROOT = "/content/datasets/so101_pick_stylus"

DRIVE_CHECKPOINT_ROOT = os.path.join(PROJECT_DIR, "output_fc/checkpoints")
LOCAL_CHECKPOINT_ROOT = "/content/output_fc/checkpoints"
CHECKPOINT_ROOT = DRIVE_CHECKPOINT_ROOT
LATEST_DIR = os.path.join(DRIVE_CHECKPOINT_ROOT, "latest")
os.makedirs(DRIVE_CHECKPOINT_ROOT, exist_ok=True)
os.makedirs(LOCAL_CHECKPOINT_ROOT, exist_ok=True)
AUTO_RESUME = True  # 若要從頭重訓，請改為 False 或清空 output_fc/checkpoints

REQUIRED_CKPT_FILES = (
    "training_state.pt",
    "config.json",
    "model.pt",
)

if not os.path.isdir(DS_ROOT):
    import zipfile
    if not os.path.isfile(DS_ZIP):
        raise FileNotFoundError(f"找不到資料集壓縮檔：{DS_ZIP}")
    print(f"Extracting {DS_ZIP} to /content ...")
    with zipfile.ZipFile(DS_ZIP, "r") as zf:
        zf.extractall("/content")
    print("Extraction complete:", DS_ROOT)

def verify_checkpoint_dir(ckpt_dir):
    """確認 checkpoint 目錄內必要檔案皆存在且大小不為 0。"""
    if not os.path.isdir(ckpt_dir):
        return False
    for fname in REQUIRED_CKPT_FILES:
        fpath = os.path.join(ckpt_dir, fname)
        if not os.path.isfile(fpath) or os.path.getsize(fpath) == 0:
            return False
    return True

def recover_atomic_checkpoints(checkpoint_root):
    """若前次原子寫入中途被強制終止，清理暫存目錄並還原備份。"""
    if not os.path.isdir(checkpoint_root):
        return
    for name in os.listdir(checkpoint_root):
        full_path = os.path.join(checkpoint_root, name)
        if name.startswith("_") and (name.endswith("_tmp") or "_tmp_" in name):
            shutil.rmtree(full_path, ignore_errors=True)
    for name in sorted(os.listdir(checkpoint_root), reverse=True):
        if name.startswith("_") and (name.endswith("_old") or "_old_" in name):
            orig_name = name[1:].rsplit("_old", 1)[0]
            old_dir = os.path.join(checkpoint_root, name)
            orig_dir = os.path.join(checkpoint_root, orig_name)
            if not os.path.exists(orig_dir):
                os.rename(old_dir, orig_dir)
            else:
                shutil.rmtree(old_dir, ignore_errors=True)

def find_latest_checkpoint(*checkpoint_roots):
    """掃描所有完整 checkpoint，回傳 step 數最高的 (目錄路徑, training_state dict)。"""
    best_dir, best_ckpt, best_step = None, None, -1
    for root in checkpoint_roots:
        if not os.path.isdir(root):
            continue
        for name in os.listdir(root):
            if name.startswith("_"):
                continue
            cand_dir = os.path.join(root, name)
            if not verify_checkpoint_dir(cand_dir):
                continue
            try:
                ckpt = torch.load(os.path.join(cand_dir, "training_state.pt"), map_location="cpu", weights_only=False)
                ckpt_step = int(ckpt.get("step", -1))
                if ckpt_step > best_step:
                    best_step = ckpt_step
                    best_dir = cand_dir
                    best_ckpt = ckpt
            except Exception as e:
                print(f"Warning: skipping corrupted checkpoint {cand_dir}: {e}")
    return best_dir, best_ckpt

recover_atomic_checkpoints(DRIVE_CHECKPOINT_ROOT)
recover_atomic_checkpoints(LOCAL_CHECKPOINT_ROOT)

# 使用 LeRobotDatasetMetadata 輕量讀取 FPS，再載入帶有 Action Chunk 的資料集
ds_meta = LeRobotDatasetMetadata(REPO_ID, root=DS_ROOT)
FPS = ds_meta.fps
CHUNK = 50
delta_timestamps = {"action": [i / FPS for i in range(CHUNK)]}
ds = LeRobotDataset(REPO_ID, root=DS_ROOT, delta_timestamps=delta_timestamps, video_backend="pyav")
print(f"Loaded dataset: {ds.num_episodes} episodes, {ds.num_frames} frames @ {FPS} FPS")
print("Dataset cameras:", ds.meta.camera_keys)

def _extract_stat_tensor(stats, key, stat_name, default_val, shape):
    if stats is not None and key in stats and stat_name in stats[key]:
        t = torch.as_tensor(stats[key][stat_name], dtype=torch.float32)
        return t.reshape(shape)
    return torch.full(shape, float(default_val), dtype=torch.float32)


class FCActionPolicy(nn.Module):
    """使用純 Fully Connected Network (MLP) 的 Action Chunking Policy。"""

    def __init__(
        self,
        state_dim,
        action_dim,
        chunk_size,
        camera_keys,
        image_size=(32, 32),
        use_images=True,
        hidden_dims=(1024, 512, 512),
        dropout=0.1,
        stats=None,
    ):
        super().__init__()
        self.state_dim = int(state_dim)
        self.action_dim = int(action_dim)
        self.chunk_size = int(chunk_size)
        self.camera_keys = list(camera_keys)
        self.image_size = tuple(image_size)
        self.use_images = bool(use_images)
        self.hidden_dims = list(hidden_dims)
        self.dropout = float(dropout)

        # 註冊 MEAN_STD 正規化統計量，確保推論時可自動還原實體關節角度
        self.register_buffer(
            "state_mean",
            _extract_stat_tensor(stats, "observation.state", "mean", 0.0, (1, self.state_dim)),
        )
        self.register_buffer(
            "state_std",
            _extract_stat_tensor(stats, "observation.state", "std", 1.0, (1, self.state_dim)).clamp(min=1e-6),
        )
        self.register_buffer(
            "action_mean",
            _extract_stat_tensor(stats, "action", "mean", 0.0, (1, 1, self.action_dim)),
        )
        self.register_buffer(
            "action_std",
            _extract_stat_tensor(stats, "action", "std", 1.0, (1, 1, self.action_dim)).clamp(min=1e-6),
        )

        for idx, cam in enumerate(self.camera_keys):
            cam_mean = _extract_stat_tensor(stats, cam, "mean", 0.5, (1, 3, 1, 1))
            cam_std = _extract_stat_tensor(stats, cam, "std", 0.5, (1, 3, 1, 1)).clamp(min=1e-6)
            self.register_buffer(f"img_mean_{idx}", cam_mean)
            self.register_buffer(f"img_std_{idx}", cam_std)

        img_flat_dim = (
            len(self.camera_keys) * 3 * self.image_size[0] * self.image_size[1]
            if self.use_images
            else 0
        )
        in_dim = self.state_dim + img_flat_dim

        layers = []
        cur_dim = in_dim
        for h_dim in self.hidden_dims:
            layers.append(nn.Linear(cur_dim, h_dim))
            layers.append(nn.LayerNorm(h_dim))
            layers.append(nn.GELU())
            if self.dropout > 0:
                layers.append(nn.Dropout(self.dropout))
            cur_dim = h_dim
        layers.append(nn.Linear(cur_dim, self.chunk_size * self.action_dim))
        self.net = nn.Sequential(*layers)

    def get_config(self):
        return {
            "arch": "fc_mlp",
            "state_dim": self.state_dim,
            "action_dim": self.action_dim,
            "chunk_size": self.chunk_size,
            "camera_keys": self.camera_keys,
            "image_size": list(self.image_size),
            "use_images": self.use_images,
            "hidden_dims": self.hidden_dims,
            "dropout": self.dropout,
        }

    def normalize_state(self, state):
        return (state - self.state_mean) / self.state_std

    def normalize_action(self, action):
        return (action - self.action_mean) / self.action_std

    def unnormalize_action(self, norm_action):
        return norm_action * self.action_std + self.action_mean

    def _encode_inputs(self, batch):
        dev = self.state_mean.device
        state = batch["observation.state"].to(device=dev, dtype=torch.float32)
        if state.ndim == 1:
            state = state.unsqueeze(0)
        elif state.ndim == 3:
            state = state[:, -1]
        feats = [self.normalize_state(state)]

        if self.use_images:
            for idx, cam in enumerate(self.camera_keys):
                raw_img = batch[cam]
                is_uint8 = raw_img.dtype == torch.uint8
                img = raw_img.to(device=dev, dtype=torch.float32)
                if img.ndim == 3:
                    img = img.unsqueeze(0)
                elif img.ndim == 5:
                    img = img[:, -1]
                if is_uint8:
                    img = img / 255.0
                img = F.interpolate(img, size=self.image_size, mode="bilinear", align_corners=False)
                img_mean = getattr(self, f"img_mean_{idx}")
                img_std = getattr(self, f"img_std_{idx}")
                img = (img - img_mean) / img_std
                feats.append(img.flatten(start_dim=1))

        return torch.cat(feats, dim=-1)

    def predict_normalized(self, batch):
        x = self._encode_inputs(batch)
        out = self.net(x)
        return out.view(-1, self.chunk_size, self.action_dim)

    @torch.no_grad()
    def predict_action(self, batch):
        was_training = self.training
        self.eval()
        try:
            pred_norm = self.predict_normalized(batch)
            return self.unnormalize_action(pred_norm)
        finally:
            self.train(was_training)

    def forward(self, batch):
        pred_norm = self.predict_normalized(batch)
        target = batch["action"].to(device=pred_norm.device, dtype=torch.float32)
        target_norm = self.normalize_action(target)

        loss = F.mse_loss(pred_norm, target_norm, reduction="none")
        action_is_pad = batch.get("action_is_pad")
        if action_is_pad is not None:
            valid_mask = (~action_is_pad.to(device=pred_norm.device, dtype=torch.bool)).unsqueeze(-1)
            loss = (loss * valid_mask).sum() / valid_mask.expand_as(loss).sum().clamp(min=1)
        else:
            loss = loss.mean()
        return {"loss": loss}


# 模型超參數設定（若需改為純關節角度輸入，可將 FC_USE_IMAGES 設為 False）
FC_USE_IMAGES = True
FC_IMAGE_SIZE = (32, 32)
FC_HIDDEN_DIMS = [1024, 512, 512]
FC_DROPOUT = 0.1

state_dim = int(ds.meta.shapes["observation.state"][0])
action_dim = int(ds.meta.shapes["action"][0])
camera_keys = list(ds.meta.camera_keys)

resume_dir, resume_ckpt = (
    find_latest_checkpoint(DRIVE_CHECKPOINT_ROOT, LOCAL_CHECKPOINT_ROOT)
    if AUTO_RESUME else (None, None)
)

if resume_dir is not None:
    print(f"--> Resuming from checkpoint: {resume_dir} (step {resume_ckpt['step']})")
    with open(os.path.join(resume_dir, "config.json"), "r", encoding="utf-8") as f:
        model_cfg = json.load(f)
    policy = FCActionPolicy(
        state_dim=model_cfg["state_dim"],
        action_dim=model_cfg["action_dim"],
        chunk_size=model_cfg["chunk_size"],
        camera_keys=model_cfg["camera_keys"],
        image_size=tuple(model_cfg["image_size"]),
        use_images=model_cfg["use_images"],
        hidden_dims=model_cfg["hidden_dims"],
        dropout=model_cfg["dropout"],
        stats=None,
    ).to(device)
    state_dict = torch.load(os.path.join(resume_dir, "model.pt"), map_location=device, weights_only=True)
    policy.load_state_dict(state_dict, strict=True)
else:
    print("--> Starting fresh training with FCActionPolicy (MLP)")
    policy = FCActionPolicy(
        state_dim=state_dim,
        action_dim=action_dim,
        chunk_size=CHUNK,
        camera_keys=camera_keys,
        image_size=FC_IMAGE_SIZE,
        use_images=FC_USE_IMAGES,
        hidden_dims=FC_HIDDEN_DIMS,
        dropout=FC_DROPOUT,
        stats=ds.meta.stats,
    ).to(device)

print("Policy input cameras:", policy.camera_keys, "| use_images:", policy.use_images)
print("FCActionPolicy ready:", round(sum(p.numel() for p in policy.parameters()) / 1e6, 2), "M params")

# 執行完整訓練與雲端斷點儲存（支援自動同步至 Drive 並結束工作階段）
- 使用 AdamW（`lr=3e-4`、`weight_decay=1e-4`）、Linear Warmup + Cosine Decay LR Scheduler 與梯度裁剪（`GRAD_CLIP_NORM=10.0`）。
- 先寫入本地暫存 `/content/output_fc/checkpoints/_ckpt_staging` 並驗證，再原子同步至 Google Drive `output_fc/checkpoints/latest` 與里程碑 `fc_pick_step_{step}`。
- 訓練完成後會繪製 Loss 曲線、執行單筆推論形狀檢查，並自動 `drive.flush_and_unmount()` 與中斷 Colab 連線。

In [ ]:
BATCH_SIZE = 32                         # 若顯存不足可調降為 16
NUM_WORKERS = 0                         # 容器 /dev/shm 預設較小，設為 0 避免 Bus error（若 shm 足夠可改 2~4）
STEPS = 15000                           # 總訓練步數
SAVE_LATEST_FREQ = 500                  # 每 500 steps 原子更新 latest，降低單次中斷損失
MILESTONE_FREQ = 5000                   # 每 5,000 steps 保留一個里程碑 checkpoint
MAX_SESSION_MINUTES = None              # 不限制單次執行時間，直到全部 STEPS 訓練完成（若需限時可改為分鐘數）
LR = 3e-4
WEIGHT_DECAY = 1e-4
WARMUP_STEPS = 500
GRAD_CLIP_NORM = 10.0
AUTO_DISCONNECT_ON_COMPLETE = True      # 訓練完成並確認雲端存檔後，自動 flush Drive 並中斷 Colab 連線
AUTO_DISCONNECT_ON_SESSION_LIMIT = True # 達單次時間上限存檔後，自動 flush Drive 並中斷 Colab 連線

optimizer = torch.optim.AdamW(policy.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

def lr_lambda(current_step):
    if current_step < WARMUP_STEPS:
        return float(current_step + 1) / float(max(1, WARMUP_STEPS))
    progress = float(current_step - WARMUP_STEPS) / float(max(1, STEPS - WARMUP_STEPS))
    return max(0.05, 0.5 * (1.0 + math.cos(math.pi * progress)))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

start_step = 0
losses = []

if resume_ckpt is not None:
    start_step = int(resume_ckpt["step"])
    losses = list(resume_ckpt.get("losses", []))
    optimizer.load_state_dict(resume_ckpt["optimizer"])
    scheduler.load_state_dict(resume_ckpt["scheduler"])
    rng_state = resume_ckpt.get("rng_state")
    if rng_state is not None:
        random.setstate(rng_state["python"])
        np.random.set_state(rng_state["numpy"])
        torch.set_rng_state(rng_state["torch"])
        if torch.cuda.is_available() and rng_state.get("cuda") is not None:
            torch.cuda.set_rng_state_all(rng_state["cuda"])
    print(f"Restored training state at step {start_step}/{STEPS} (loaded {len(losses)} loss records)")

def sync_dir_atomically(src_dir, dst_dir):
    """以暫存目錄 + 原子更名方式將本地完整的 checkpoint 同步到雲端目標目錄。"""
    parent_dir = os.path.dirname(dst_dir)
    base_name = os.path.basename(dst_dir)
    tmp_dir = os.path.join(parent_dir, f"_{base_name}_tmp")
    old_dir = os.path.join(parent_dir, f"_{base_name}_old")
    os.makedirs(parent_dir, exist_ok=True)
    shutil.rmtree(tmp_dir, ignore_errors=True)
    shutil.rmtree(old_dir, ignore_errors=True)
    if os.path.exists(tmp_dir):
        tmp_dir = os.path.join(parent_dir, f"_{base_name}_tmp_{int(time.time() * 1000)}")
    if os.path.exists(old_dir):
        old_dir = os.path.join(parent_dir, f"_{base_name}_old_{int(time.time() * 1000)}")
    shutil.copytree(src_dir, tmp_dir, dirs_exist_ok=True)
    os.sync()
    if os.path.exists(dst_dir):
        os.rename(dst_dir, old_dir)
    os.rename(tmp_dir, dst_dir)
    shutil.rmtree(old_dir, ignore_errors=True)
    os.sync()

def save_checkpoint(target_dirs, current_step):
    """先寫入本地 SSD 暫存並驗證，再原子同步至一或多個雲端目標目錄。"""
    if isinstance(target_dirs, str):
        target_dirs = [target_dirs]
    staging_dir = os.path.join(LOCAL_CHECKPOINT_ROOT, "_ckpt_staging")
    shutil.rmtree(staging_dir, ignore_errors=True)
    os.makedirs(staging_dir, exist_ok=True)

    state_payload = {
        "step": current_step,
        "optimizer": optimizer.state_dict(),
        "scheduler": scheduler.state_dict(),
        "losses": losses,
        "rng_state": {
            "python": random.getstate(),
            "numpy": np.random.get_state(),
            "torch": torch.get_rng_state(),
            "cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,
        },
    }
    try:
        with open(os.path.join(staging_dir, "config.json"), "w", encoding="utf-8") as f:
            json.dump(policy.get_config(), f, indent=2, ensure_ascii=False)
        torch.save(policy.state_dict(), os.path.join(staging_dir, "model.pt"))
        torch.save(state_payload, os.path.join(staging_dir, "training_state.pt"))
        if not verify_checkpoint_dir(staging_dir):
            raise RuntimeError(f"本地暫存 checkpoint 驗證失敗：{staging_dir}")

        for dst_dir in target_dirs:
            sync_dir_atomically(staging_dir, dst_dir)
            if not verify_checkpoint_dir(dst_dir):
                raise RuntimeError(f"雲端 checkpoint 驗證失敗：{dst_dir}")
    finally:
        shutil.rmtree(staging_dir, ignore_errors=True)

def build_trimmed_subset(dataset, ds_root, move_thresh=3.0, keep_lead_frames=10):
    """跳過每條 episode 開頭尚未動作的靜止等待幀，避免學到起點不動的固定點。"""
    files = sorted(glob.glob(f"{ds_root}/data/*/*.parquet"))
    df = pd.concat([pd.read_parquet(f) for f in files], ignore_index=True)
    states = np.stack(df["observation.state"].values)
    eps = df["episode_index"].values
    indices = df["index"].values

    valid_indices = []
    for ep in np.unique(eps):
        m = eps == ep
        ep_states = states[m]
        ep_idx = indices[m]
        diffs = np.max(np.abs(ep_states - ep_states[0]), axis=1)
        moved = np.where(diffs > move_thresh)[0]
        start_offset = max(0, int(moved[0]) - keep_lead_frames) if len(moved) > 0 else 0
        valid_indices.extend(ep_idx[start_offset:].tolist())

    removed = len(df) - len(valid_indices)
    print(
        f"Trimmed idle prefix: {len(df)} -> {len(valid_indices)} frames "
        f"(removed {removed}, {removed / len(df) * 100:.1f}%)"
    )
    return torch.utils.data.Subset(dataset, valid_indices)

exit_reason = None
last_completed_step = start_step

if start_step >= STEPS:
    print(f"Training already completed ({start_step}/{STEPS} steps). Skipping loop.")
    exit_reason = "completed"
else:
    train_ds = build_trimmed_subset(ds, DS_ROOT, move_thresh=3.0, keep_lead_frames=10)
    loader = torch.utils.data.DataLoader(
        train_ds,
        batch_size=BATCH_SIZE,
        shuffle=True,
        drop_last=True,
        num_workers=NUM_WORKERS,
        pin_memory=(device == "cuda" and NUM_WORKERS > 0),
    )
    policy.train()
    it = iter(loader)
    session_start_t = time.time()

    try:
        for step in range(start_step + 1, STEPS + 1):
            try:
                batch = next(it)
            except StopIteration:
                it = iter(loader)
                batch = next(it)

            out = policy(batch)
            loss = out["loss"]

            optimizer.zero_grad()
            loss.backward()
            if GRAD_CLIP_NORM and GRAD_CLIP_NORM > 0:
                torch.nn.utils.clip_grad_norm_(policy.parameters(), GRAD_CLIP_NORM)
            optimizer.step()
            scheduler.step()

            loss_val = loss.detach().item()
            losses.append(loss_val)
            last_completed_step = step
            elapsed_min = (time.time() - session_start_t) / 60.0

            if step % 200 == 0 or step == start_step + 1:
                cur_lr = optimizer.param_groups[0]["lr"]
                print(f"step {step:5d}/{STEPS} | loss {loss_val:.4f} | lr {cur_lr:.2e} | elapsed {elapsed_min:.1f}m")

            reach_limit = (
                MAX_SESSION_MINUTES is not None
                and elapsed_min >= MAX_SESSION_MINUTES
                and step < STEPS
            )
            save_targets = []
            if step % SAVE_LATEST_FREQ == 0 or step == STEPS or reach_limit:
                save_targets.append(LATEST_DIR)
            if step % MILESTONE_FREQ == 0 or step == STEPS:
                save_targets.append(os.path.join(DRIVE_CHECKPOINT_ROOT, f"fc_pick_step_{step}"))

            if save_targets:
                save_checkpoint(save_targets, step)
                for t_dir in save_targets:
                    print(f"--> Checkpoint synced & verified at step {step}: {t_dir}")

            if reach_limit:
                exit_reason = "session_limit"
                print(
                    f"\n[Session Limit] Reached {elapsed_min:.1f}m (>= {MAX_SESSION_MINUTES}m). "
                    f"Saved progress at step {step}/{STEPS} to {LATEST_DIR}."
                )
                break
        else:
            exit_reason = "completed"
    except KeyboardInterrupt:
        exit_reason = "interrupted"
        if last_completed_step > start_step:
            save_checkpoint([LATEST_DIR], last_completed_step)
            print(f"\n[Interrupted] Saved & verified progress at step {last_completed_step}/{STEPS} to {LATEST_DIR}.")
    except Exception as e:
        exit_reason = "error"
        if last_completed_step > start_step:
            try:
                save_checkpoint([LATEST_DIR], last_completed_step)
                print(f"\n[Error Recovery] Saved progress at step {last_completed_step}/{STEPS} to {LATEST_DIR}.")
            except Exception as save_err:
                print(f"\n[Error Recovery] Emergency save failed: {save_err}")
        raise

if losses:
    plt.figure(figsize=(8, 3))
    plt.plot(range(1, len(losses) + 1), losses, alpha=0.7)
    plt.xlabel("step")
    plt.ylabel("masked MSE loss (normalized)")
    plt.title("FC (MLP) Training Loss (SO-101 Pick Stylus)")
    plt.yscale("log")
    plt.grid(True)
    plt.show()

# 推論輸出檢查（驗證 predict_action 回傳形狀與實際關節角度範圍）
policy.eval()
sample = ds[0]
sample_batch = {
    k: (v.unsqueeze(0) if isinstance(v, torch.Tensor) else v)
    for k, v in sample.items()
}
pred_chunk = policy.predict_action(sample_batch)
print("Sanity check - predicted action chunk shape:", tuple(pred_chunk.shape))
print("Sanity check - first predicted step:", pred_chunk[0, 0].detach().cpu().numpy().round(3))

if exit_reason in ("completed", "session_limit"):
    final_milestone = os.path.join(DRIVE_CHECKPOINT_ROOT, f"fc_pick_step_{STEPS}")
    verify_target = (
        final_milestone if (exit_reason == "completed" and verify_checkpoint_dir(final_milestone))
        else LATEST_DIR
    )
    if not verify_checkpoint_dir(verify_target):
        raise RuntimeError(f"雲端最終檢查失敗，找不到完整模型檔案：{verify_target}")

    weights_mb = os.path.getsize(os.path.join(verify_target, "model.pt")) / (1024 * 1024)
    print(f"[Cloud Verified] 模型已完整儲存至 Google Drive：{verify_target} (model.pt: {weights_mb:.1f} MB)")

    should_disconnect = (
        (exit_reason == "completed" and AUTO_DISCONNECT_ON_COMPLETE)
        or (exit_reason == "session_limit" and AUTO_DISCONNECT_ON_SESSION_LIMIT)
    )
    if should_disconnect:
        os.sync()
        try:
            from google.colab import drive, runtime
            print("[Cloud Sync] Flushing Google Drive writes before disconnecting...")
            drive.flush_and_unmount()
            print("[Runtime] Disconnecting Colab runtime now...")
            runtime.unassign()
        except ImportError:
            print("[Runtime] 非 Colab 環境，略過 flush_and_unmount / runtime.unassign。")